<a href="https://colab.research.google.com/github/KP-365/Skinrash-detection/blob/main/Training_modelafterparamtuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from datasets import load_dataset
import numpy as np

SEED = 1337
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 8
BATCH_SIZE = 32
IMG_SIZE = 224
EPOCHS_HEAD = 100
EPOCHS_FINETUNE = 50
LR_HEAD = 1e-3
LR_FINETUNE = 1e-4      # winning value from grid search
DROPOUT_P = 0.3         # winning value from grid search
PATIENCE = 7

print(f"Using device: {DEVICE}")

Using device: cuda


In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.should_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

In [ ]:
clean = load_dataset("eceunal/bug-bite-images-hf")
labels = clean["train"].features["label"].names
print("Classes:", labels)
print(f"Train: {len(clean['train'])} | Validation: {len(clean['validation'])} | Test: {len(clean['test'])}")

Classes: ['ants', 'bed_bugs', 'chiggers', 'fleas', 'mosquitos', 'no_bites', 'spiders', 'ticks']
Train: 896 | Validation: 106 | Test: 53


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(30),
    transforms.RandomAffine(degrees=15, translate=(0.15, 0.15), scale=(0.8, 1.2), shear=10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomAutocontrast(p=0.3),
    transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.3),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
class BugBiteDataset(Dataset):
    def __init__(self, hf_split, transform):
        self.data = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex = self.data[idx]
        img = ex["image"].convert("RGB")
        img = self.transform(img)
        return img, ex["label"]

train_ds = BugBiteDataset(clean["train"], train_transform)
val_ds = BugBiteDataset(clean["validation"], eval_transform)
test_ds = BugBiteDataset(clean["test"], eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
def build_model():
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    for param in model.features.parameters():
        param.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=DROPOUT_P),
        nn.Linear(in_features, NUM_CLASSES)
    )
    return model.to(DEVICE)

model = build_model()

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, targets in loader:
        imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == targets).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, targets in loader:
            imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, targets)
            total_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == targets).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total

criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD)
early_stopper = EarlyStopping(patience=PATIENCE)

print("\n=== Phase 1: Head-only training (frozen backbone) ===")
best_val_acc = 0
best_epoch_phase1 = 0
for epoch in range(EPOCHS_HEAD):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{EPOCHS_HEAD} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch_phase1 = epoch + 1
        torch.save(model.state_dict(), "best_model_tuned_phase1.pt")
        print(f"  -> New best (epoch {epoch+1}), checkpoint saved.")

    early_stopper(val_loss)
    if early_stopper.should_stop:
        print(f"Early stopping triggered at epoch {epoch+1} (phase 1)")
        break

print(f"\nPhase 1 complete. Best val_acc={best_val_acc:.4f} at epoch {best_epoch_phase1}")


=== Phase 1: Head-only training (frozen backbone) ===
Epoch 1/100 | train_loss=1.9576 train_acc=0.2377 | val_loss=1.8786 val_acc=0.3208
  -> New best (epoch 1), checkpoint saved.
Epoch 2/100 | train_loss=1.7613 train_acc=0.3973 | val_loss=1.7816 val_acc=0.3585
  -> New best (epoch 2), checkpoint saved.
Epoch 3/100 | train_loss=1.6476 train_acc=0.4286 | val_loss=1.7042 val_acc=0.3868
  -> New best (epoch 3), checkpoint saved.
Epoch 4/100 | train_loss=1.5423 train_acc=0.4777 | val_loss=1.6565 val_acc=0.4340
  -> New best (epoch 4), checkpoint saved.
Epoch 5/100 | train_loss=1.4838 train_acc=0.4888 | val_loss=1.6211 val_acc=0.4151
Epoch 6/100 | train_loss=1.4441 train_acc=0.4911 | val_loss=1.6060 val_acc=0.4151
Epoch 7/100 | train_loss=1.3908 train_acc=0.5469 | val_loss=1.5918 val_acc=0.4151
Epoch 8/100 | train_loss=1.3711 train_acc=0.5379 | val_loss=1.5895 val_acc=0.4057
Epoch 9/100 | train_loss=1.3378 train_acc=0.5536 | val_loss=1.5524 val_acc=0.4340
Epoch 10/100 | train_loss=1.3210 tr

In [ ]:
print("\n=== Phase 2: Fine-tuning last 3 blocks ===")
model.load_state_dict(torch.load("best_model_tuned_phase1.pt", weights_only=False))

for param in model.features[-3:].parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FINETUNE)
early_stopper = EarlyStopping(patience=PATIENCE)

best_val_acc = 0
best_epoch_phase2 = 0
for epoch in range(EPOCHS_FINETUNE):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = eval_epoch(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{EPOCHS_FINETUNE} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch_phase2 = epoch + 1
        torch.save(model.state_dict(), "best_model_tuned_final.pt")
        print(f"  -> New best (epoch {epoch+1}), checkpoint saved.")

    early_stopper(val_loss)
    if early_stopper.should_stop:
        print(f"Early stopping triggered at epoch {epoch+1} (phase 2)")
        break

print(f"\nPhase 2 complete. Best val_acc={best_val_acc:.4f} at epoch {best_epoch_phase2}")


=== Phase 2: Fine-tuning last 3 blocks ===
Epoch 1/50 | train_loss=1.1736 train_acc=0.5871 | val_loss=1.4489 val_acc=0.4811
  -> New best (epoch 1), checkpoint saved.
Epoch 2/50 | train_loss=1.0495 train_acc=0.6317 | val_loss=1.3737 val_acc=0.5283
  -> New best (epoch 2), checkpoint saved.
Epoch 3/50 | train_loss=0.9901 train_acc=0.6484 | val_loss=1.3630 val_acc=0.5189
Epoch 4/50 | train_loss=0.8940 train_acc=0.6797 | val_loss=1.4221 val_acc=0.5283
Epoch 5/50 | train_loss=0.8924 train_acc=0.6708 | val_loss=1.3333 val_acc=0.5377
  -> New best (epoch 5), checkpoint saved.
Epoch 6/50 | train_loss=0.8903 train_acc=0.6719 | val_loss=1.3380 val_acc=0.5472
  -> New best (epoch 6), checkpoint saved.
Epoch 7/50 | train_loss=0.8609 train_acc=0.6942 | val_loss=1.3446 val_acc=0.5377
Epoch 8/50 | train_loss=0.7820 train_acc=0.7266 | val_loss=1.2707 val_acc=0.5566
  -> New best (epoch 8), checkpoint saved.
Epoch 9/50 | train_loss=0.8157 train_acc=0.7109 | val_loss=1.2968 val_acc=0.5755
  -> New bes

In [ ]:
model.load_state_dict(torch.load("best_model_tuned_final.pt", weights_only=False))
test_loss, test_acc = eval_epoch(model, test_loader, criterion)
print(f"\nFinal TEST accuracy: {test_acc:.4f} | test_loss: {test_loss:.4f}")


Final TEST accuracy: 0.7547 | test_loss: 0.8318


In [ ]:
from google.colab import files
files.download("best_model_tuned_final.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>